# BANTAI — Pretrained pothole evaluation (v1)
**Evaluation only. No fine-tuning, training, or changes to your app.**

Upload this notebook to Colab. Select a T4 GPU if available and run sections in order.
We use PeterHdd's pothole YOLOv8s and a fixed sample of **200 pothole images + 100
images with no annotated potholes** from RDD2022 India. This deliberately balanced
sample is a diagnostic test, not an estimate of how often potholes occur in traffic.

Dataset/model work stays on Colab's temporary SSD (`/content`). Results are backed up
to Drive. A new runtime requires downloading/rebuilding the data again; completed
per-image predictions are restored from Drive. An interrupted mAP pass restarts that
short pass. Download the results ZIP in section 7 and send it back in our chat.

**Maintenance:** edit the existing section that fails, then rerun it and its dependent
sections. Keep one configuration cell. Use a new RUN_NAME when settings change.

Sources: [model](https://huggingface.co/peterhdd/pothole-detection-yolov8),
[RDD2022 official release](https://github.com/sekilab/RoadDamageDetector#dataset),
[RDD2022 paper](https://doi.org/10.1002/gdj3.260).
Training-data overlap has not been independently ruled out. Local Philippine camera
performance and physical pothole depth/severity are not measured by this notebook.


## 1. Settings and persistent results
Run this first. Allow the Drive mount so completed predictions survive runtime loss.
The confidence threshold is fixed before evaluation; do not tune it on these results
and describe the rerun as an untouched test. Use a separate validation set for tuning.


In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, json, random, shutil, sys, subprocess, os

RUN_NAME = "peterhdd_rdd_india_eval_001"
SEED = 42
POSITIVE_IMAGES = 200
BACKGROUND_IMAGES = 100
IMAGE_SIZE = 640
CONFIDENCE = 0.25
MATCH_IOU = 0.50
MAP_SCORE_FLOOR = 0.001
ULTRALYTICS_VERSION = "8.3.221"
MODEL_REVISION = "da7747e"  # Fixed Hugging Face commit, never a moving 'main'.
MODEL_REPO = "peterhdd/pothole-detection-yolov8"

assert RUN_NAME.replace('_', '').isalnum(), "Use letters, numbers and underscores."
assert POSITIVE_IMAGES > 0 and BACKGROUND_IMAGES > 0
assert 0 < CONFIDENCE < 1 and 0 < MATCH_IOU <= 1
ROOT = Path('/content/bantai_pretrained') / RUN_NAME
OUT = ROOT / 'results'
drive.mount('/content/drive')
BACKUP = Path('/content/drive/MyDrive/BANTAI/pretrained_evaluation') / RUN_NAME
for folder in (ROOT, OUT, BACKUP):
    folder.mkdir(parents=True, exist_ok=True)
config = dict(version='pretrained-eval-v1', seed=SEED, positive=POSITIVE_IMAGES,
              background=BACKGROUND_IMAGES, imgsz=IMAGE_SIZE, confidence=CONFIDENCE,
              match_iou=MATCH_IOU, map_score_floor=MAP_SCORE_FLOOR,
              ultralytics=ULTRALYTICS_VERSION, repo=MODEL_REPO, revision=MODEL_REVISION)
config_id = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
for previous in (OUT / 'config.json', BACKUP / 'config.json'):
    if previous.exists():
        assert json.loads(previous.read_text()) == config, 'Settings changed: choose a new RUN_NAME.'
(OUT / 'config.json').write_text(json.dumps(config, indent=2))
shutil.copy2(OUT / 'config.json', BACKUP / 'config.json')
print('Run:', RUN_NAME, '| results backup:', BACKUP)


## 2. Install the evaluation package
Running this cell installs the pinned Ultralytics package into this Colab runtime.
It reuses Colab's PyTorch/GPU setup. Transitive versions depend on Colab; the complete
installed environment is exported with the results. No packages are installed on your PC.
If Colab requests a runtime restart, restart and rerun from section 1.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'ultralytics==' + ULTRALYTICS_VERSION], check=True)
os.environ['YOLO_CONFIG_DIR'] = str(ROOT / 'ultralytics_settings')
import torch
import ultralytics
from ultralytics import YOLO, settings
from PIL import Image, ImageDraw, ImageOps
settings.update({'sync': False})
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Ultralytics:', ultralytics.__version__, '| PyTorch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU (slower, still usable)')
(OUT / 'environment.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True))


## 3. Download the pretrained model and India dataset
The dataset is approximately 502 MB. Only the labeled `train` folders are extracted;
RDD's official challenge test folders have no labels and cannot produce local scores.
We are evaluating an existing model, so using RDD's labeled `train` folder does not
train anything. No Kaggle/Roboflow account is required.

If the original S3 link is unavailable, use the official Figshare mirror and retrieve
only the nested India ZIP. A server that refuses partial downloads stops safely.
Incomplete downloads restart when this section is rerun.


In [ ]:
import collections, io, math, re, stat, uuid, zipfile
import urllib.request, urllib.error
import xml.etree.ElementTree as ET
from pathlib import PurePosixPath
VERSION = 'pretrained-eval-v1'
COUNTRIES = ('India',)
BASE_URL = 'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/'
URLS = {'India': BASE_URL + 'RDD2022_India.zip'}
FIGSHARE_ARTICLE_API = 'https://api.figshare.com/v2/articles/21431547'
FIGSHARE_FILENAME = 'RDD2022_released_through_CRDDC2022.zip'

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def identity(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

def atomic_copy(source, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".partial")
    shutil.copyfile(source, temporary)
    temporary.replace(destination)

def backup_tree(source, destination):
    source = Path(source)
    for path in sorted(source.rglob("*")):
        if path.is_file() and not path.name.endswith(".partial"):
            atomic_copy(path, Path(destination) / path.relative_to(source))

def extract_training(archive, destination):
    """Never extract unlabeled official test images, links, or escaping paths."""
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    count = 0
    with zipfile.ZipFile(archive) as bundle:
        for entry in bundle.infolist():
            relative = PurePosixPath(entry.filename)
            if (relative.is_absolute() or ".." in relative.parts or "\\" in entry.filename
                    or any(":" in part for part in relative.parts)
                    or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError("Unsafe archive member")
            if entry.is_dir() or "train" not in relative.parts:
                continue
            if relative.suffix.lower() not in {".jpg", ".jpeg", ".png", ".xml"}:
                continue
            target = destination.joinpath(*relative.parts).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Archive path escapes extraction directory")
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists():
                # Re-extraction after interruption verifies bytes without replacing originals.
                with bundle.open(entry) as stream:
                    expected = hashlib.sha256(stream.read()).hexdigest()
                if sha256(target) != expected:
                    raise ValueError("Extracted source changed: " + target.name)
            else:
                temporary = target.with_name(target.name + ".partial")
                with bundle.open(entry) as source, temporary.open("wb") as output:
                    shutil.copyfileobj(source, output)
                temporary.replace(target)
            count += 1
    if count == 0:
        raise ValueError("Archive contains no recognized training files")
    return count

class RangeDownload(io.RawIOBase):
    """Seekable HTTP ZIP reader; cache one 8 MiB block, never fetch the full archive."""
    def __init__(self, url, size, block_size=8 * 1024**2):
        self.url, self.size, self.block_size = url, size, block_size
        self.position, self.block_start, self.block = 0, -1, b""
        self.transferred_bytes = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.position

    def seek(self, offset, whence=0):
        position = offset + (0 if whence == 0 else self.position if whence == 1 else self.size)
        if whence not in (0, 1, 2) or position < 0:
            raise ValueError("Invalid archive seek")
        self.position = position
        return position

    def read(self, size=-1):
        remaining = min(self.size - self.position, size if size >= 0 else self.size)
        output = bytearray()
        while remaining > 0:
            start = self.position // self.block_size * self.block_size
            if start != self.block_start:
                end = min(start + self.block_size, self.size) - 1
                # Fresh API redirects avoid expired cached signed storage URLs.
                url = self.url + "?request=" + uuid.uuid4().hex
                request = urllib.request.Request(url, headers={"Range": f"bytes={start}-{end}",
                                                               "Accept-Encoding": "identity"})
                with urllib.request.urlopen(request, timeout=120) as response:
                    expected = f"bytes {start}-{end}/{self.size}"
                    if response.status != 206 or response.headers.get("Content-Range") != expected:
                        raise IOError("Mirror did not honor a partial request; full archive download refused")
                    data = response.read(end - start + 2)
                    if len(data) != end - start + 1:
                        raise IOError("Incomplete archive range; rerun Phase 1")
                self.block_start, self.block = start, data
                self.transferred_bytes += len(data)
            offset = self.position - start
            count = min(remaining, len(self.block) - offset)
            output.extend(self.block[offset:offset + count])
            self.position += count
            remaining -= count
        return bytes(output)

def mirror_country(country, archive):
    """Stream the original country ZIP nested inside Figshare's full RDD archive."""
    with urllib.request.urlopen(FIGSHARE_ARTICLE_API, timeout=60) as response:
        article = json.load(response)
    files = [file for file in article["files"] if file["name"] == FIGSHARE_FILENAME]
    if len(files) != 1:
        raise ValueError("Official mirror archive not uniquely identified")
    descriptor = files[0]
    file_id = int(descriptor["id"])
    size = int(descriptor["size"])
    if size <= 0:
        raise ValueError("Invalid mirror archive size")
    url = "https://api.figshare.com/v2/file/download/" + str(file_id)
    temporary = Path(archive).with_suffix(".partial")
    print("Using official Figshare mirror for", country, "(country ZIP only)", flush=True)
    with RangeDownload(url, size) as remote, zipfile.ZipFile(remote) as source:
        selected = [entry for entry in source.infolist()
                    if PurePosixPath(entry.filename).name == country + ".zip" and not entry.is_dir()]
        if len(selected) != 1:
            raise ValueError("Mirror archive country ZIP not uniquely identified")
        entry = selected[0]
        if entry.compress_type != zipfile.ZIP_STORED:
            raise ValueError("Mirror layout changed; expected directly range-readable country ZIP")
        downloaded = 0
        with source.open(entry) as stream, temporary.open("wb") as output:
            for chunk in iter(lambda: stream.read(4 * 1024**2), b""):
                output.write(chunk)
                downloaded += len(chunk)
                if downloaded // (64 * 1024**2) != (downloaded - len(chunk)) // (64 * 1024**2):
                    print(country, round(downloaded / 1024**2), "MiB from mirror", flush=True)
        if downloaded != entry.file_size or not zipfile.is_zipfile(temporary):
            raise IOError("Incomplete country ZIP from mirror; rerun Phase 1")
        origin = {"url": url, "article_api": FIGSHARE_ARTICLE_API, "file_id": file_id,
                  "source_filename": FIGSHARE_FILENAME, "source_archive_bytes": size,
                  "source_archive_md5": descriptor.get("computed_md5"),
                  "integrity": "Original member CRC verified; full source archive MD5 not verified",
                  "local_archive": "Original nested country ZIP; image/XML contents unchanged",
                  "source_member": entry.filename, "downloaded_range_bytes": remote.transferred_bytes}
    write_json(Path(archive).with_suffix(".origin.json"), origin)
    temporary.replace(archive)
    return origin

def download_country(country, url, archive):
    print("Downloading", country, "directly to runtime storage", flush=True)
    request = urllib.request.Request(url, headers={"User-Agent": "BANTAI-academic-baseline/1"})
    temporary = archive.with_suffix(".partial")
    try:
        response = urllib.request.urlopen(request, timeout=120)
    except urllib.error.HTTPError as error:
        if error.code not in {403, 404, 410}:
            raise
        error.close()
        return mirror_country(country, archive)
    with response, temporary.open("wb") as output:
        expected = response.headers.get("Content-Length")
        downloaded = 0
        for chunk in iter(lambda: response.read(4 * 1024**2), b""):
            output.write(chunk)
            downloaded += len(chunk)
            if downloaded // (64 * 1024**2) != (downloaded - len(chunk)) // (64 * 1024**2):
                print(country, round(downloaded / 1024**2), "MiB", flush=True)
        if expected and downloaded != int(expected):
            raise IOError("Incomplete download; rerun to retry")
    if not zipfile.is_zipfile(temporary):
        raise ValueError("Download was not a ZIP archive")
    origin = {"url": url}
    write_json(archive.with_suffix(".origin.json"), origin)
    temporary.replace(archive)
    return origin

def download_sources(root):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(root).free < 8 * 1024**3:
        raise RuntimeError("Need at least 8 GiB free for archives and extracted training images")
    previous_path = root / "source_provenance.json"
    previous = read_json(previous_path) if previous_path.exists() else {}
    provenance = dict(previous)
    for country, url in URLS.items():
        archive = root / "archives" / (country + ".zip")
        archive.parent.mkdir(parents=True, exist_ok=True)
        if not archive.exists():
            origin = download_country(country, url, archive)
        else:
            origin_path = archive.with_suffix(".origin.json")
            origin = read_json(origin_path) if origin_path.exists() else {"url": url}
        digest = sha256(archive)
        if country in previous:
            if previous[country]["sha256"] != digest:
                raise ValueError("Cached archive changed; use a new attempt directory")
        count = extract_training(archive, root / "raw" / country)
        provenance[country] = {**origin, "sha256": digest, "bytes": archive.stat().st_size,
                               "extracted_training_files": count}
        write_json(root / "source_provenance.json", provenance)
    return provenance

def difference_hash(image):
    from PIL import Image
    pixels = image.convert("L").resize((9, 8), Image.Resampling.LANCZOS).tobytes()
    value = 0
    for row in range(8):
        for column in range(8):
            value = (value << 1) | (pixels[row * 9 + column] > pixels[row * 9 + column + 1])
    return value

def parse_annotation(path, width, height):
    data = Path(path).read_bytes()
    if b"<!DOCTYPE" in data.upper() or b"<!ENTITY" in data.upper():
        raise ValueError("XML declarations are not allowed")
    tree = ET.fromstring(data)
    if int(tree.findtext("size/width", "0")) != width or int(tree.findtext("size/height", "0")) != height:
        raise ValueError("Annotation/image dimensions differ")
    boxes, flags = [], []
    for obj in tree.findall("object"):
        name = (obj.findtext("name") or "").strip()
        if name != "D40":
            continue
        values = [float(obj.findtext("bndbox/" + field, "nan"))
                  for field in ("xmin", "ymin", "xmax", "ymax")]
        x1, y1, x2, y2 = values
        if not all(math.isfinite(v) for v in values) or not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
            raise ValueError("Invalid pothole box; entire image quarantined")
        if obj.findtext("difficult", "0") == "1":
            flags.append("difficult pothole")
        if min(x2 - x1, y2 - y1) < 8:
            flags.append("small pothole box")
        boxes.append(values)
    # RDD pixel edges are retained as provided; no silent clipping or box enlargement.
    return sorted(boxes), sorted(set(flags))

def audit_sources(root):
    from PIL import Image, ImageOps
    root = Path(root)
    records, exclusions = [], []
    for country in COUNTRIES:
        folder = root / "raw" / country
        images = sorted(p for p in folder.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
        xmls = collections.defaultdict(list)
        for xml in sorted(folder.rglob("*.xml")):
            xmls[xml.stem].append(xml)
        for index, path in enumerate(images):
            identifier = country + "/" + path.relative_to(folder).as_posix()
            try:
                matches = xmls[path.stem]
                if len(matches) != 1:
                    raise ValueError("Missing or ambiguous annotation; never assume background")
                with Image.open(path) as image:
                    image.load()
                    width, height = image.size
                    if image.getexif().get(274, 1) != 1:
                        raise ValueError("EXIF rotation requires annotation review")
                    boxes, flags = parse_annotation(matches[0], width, height)
                    perceptual = difference_hash(ImageOps.grayscale(image))
                records.append({"id": identifier, "country": country,
                                "image": path.relative_to(root).as_posix(),
                                "xml": matches[0].relative_to(root).as_posix(),
                                "sha256": sha256(path), "xml_sha256": sha256(matches[0]),
                                "dhash": perceptual, "width": width, "height": height,
                                "boxes": boxes, "flags": flags})
            except (ValueError, OSError, ET.ParseError, Image.DecompressionBombError) as error:
                exclusions.append({"id": identifier, "reason": str(error)})
            if index % 1000 == 0:
                print(country, index, "/", len(images), "audited", flush=True)
        if not images:
            raise ValueError("No images for " + country)
    # Same image bytes with incompatible labels are never treated as reliable negatives.
    by_digest = collections.defaultdict(list)
    for record in records:
        by_digest[record["sha256"]].append(record)
    kept = []
    for group in by_digest.values():
        if len({identity(r["boxes"]) for r in group}) > 1:
            exclusions.extend({"id": r["id"], "reason": "Duplicate image with conflicting annotations"} for r in group)
            continue
        kept.append(group[0])
        exclusions.extend({"id": r["id"], "reason": "Exact duplicate", "retained": group[0]["id"]} for r in group[1:])
    records = sorted(kept, key=lambda r: r["id"])
    summary = {}
    for country in COUNTRIES:
        subset = [r for r in records if r["country"] == country]
        summary[country] = {"usable_images": len(subset), "positive_images": sum(bool(r["boxes"]) for r in subset),
                            "candidate_backgrounds": sum(not r["boxes"] for r in subset),
                            "pothole_boxes": sum(len(r["boxes"]) for r in subset)}
    report = {"version": VERSION, "records": records, "exclusions": exclusions, "summary": summary,
              "sources": read_json(root / "source_provenance.json"),
              "limitations": "Background labels may be incomplete. Phone/CCTV performance is unverified."}
    report["audit_id"] = identity(report)
    write_json(root / "inspection" / "audit.json", report)
    print(json.dumps(summary, indent=2))
    print("Quarantined/deduplicated:", len(exclusions))
    return report


In [ ]:
# Use the canonical weight file from a fixed model revision, and record its hash.
MODEL_PATH = ROOT / 'peterhdd_pothole.pt'
model_url = f'https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/best.pt'
if not MODEL_PATH.exists():
    temporary = MODEL_PATH.with_suffix('.partial')
    with urllib.request.urlopen(model_url, timeout=120) as response, temporary.open('wb') as target:
        expected = response.headers.get('Content-Length')
        total = 0
        for chunk in iter(lambda: response.read(1024 * 1024), b''):
            target.write(chunk)
            total += len(chunk)
    if expected and total != int(expected):
        raise IOError('Incomplete model download. Rerun section 3.')
    if total < 1024 * 1024:
        raise ValueError('Unexpected model download size. Check the model URL.')
    temporary.replace(MODEL_PATH)
model_digest = sha256(MODEL_PATH)
model_info = dict(repo=MODEL_REPO, revision=MODEL_REVISION, url=model_url,
                  sha256=model_digest, bytes=MODEL_PATH.stat().st_size)
for previous in (OUT / 'model.json', BACKUP / 'model.json'):
    if previous.exists():
        assert read_json(previous)['sha256'] == model_digest, 'Model changed; choose a new RUN_NAME.'
write_json(OUT / 'model.json', model_info)
# Loading this PyTorch checkpoint trusts the selected publisher's model file.
model = YOLO(str(MODEL_PATH))
assert len(model.names) == 1 and 'pothole' in str(model.names[0]).lower(), model.names
print('Loaded:', model.names)
provenance = download_sources(ROOT)
write_json(OUT / 'source_provenance.json', provenance)
backup_tree(OUT, BACKUP)


## 4. Prepare the fixed evaluation sample
Check image/XML dimensions, quarantine invalid labels, and remove exact duplicates.
Keep D40 potholes as class 0; other damage types are not potholes. Valid XML with no
D40 is treated as a background candidate. Small/difficult boxes remain included and
are flagged. Missing XML is never treated as an empty road.

Sampling is deterministic. The manifest records selected image and annotation hashes.
We do not claim route independence or absence of near duplicates. Background labels
may omit defects; inspect examples after evaluation. No sample images are shown
before the fixed evaluation.


In [ ]:
audit = audit_sources(ROOT)
records = audit['records']
rng = random.Random(SEED)
positives = [r for r in records if r['boxes']]
backgrounds = [r for r in records if not r['boxes']]
assert len(positives) >= POSITIVE_IMAGES, f'Only {len(positives)} usable positive images; review counts.'
assert len(backgrounds) >= BACKGROUND_IMAGES, f'Only {len(backgrounds)} usable backgrounds; review counts.'
selected = sorted(rng.sample(positives, POSITIVE_IMAGES) +
                  rng.sample(backgrounds, BACKGROUND_IMAGES), key=lambda r: r['id'])
manifest = dict(config_id=config_id, records=selected)
manifest['sample_id'] = identity(manifest)
for previous in (OUT / 'manifest.json', BACKUP / 'manifest.json'):
    if previous.exists():
        assert read_json(previous) == manifest, 'Sample changed; use a new RUN_NAME.'
write_json(OUT / 'manifest.json', manifest)
write_json(OUT / 'audit_summary.json', dict(summary=audit['summary'], exclusions=audit['exclusions']))
prepared = ROOT / ('sample_' + manifest['sample_id'][:12])
for folder in (prepared / 'images/val', prepared / 'labels/val'):
    folder.mkdir(parents=True, exist_ok=True)
for index, r in enumerate(selected):
    stem = f'{index:04d}'
    target = prepared / 'images/val' / (stem + Path(r['image']).suffix.lower())
    atomic_copy(ROOT / r['image'], target)
    r_width, r_height = r['width'], r['height']
    labels = []
    for x1, y1, x2, y2 in r['boxes']:
        labels.append(f'0 {(x1+x2)/2/r_width:.10f} {(y1+y2)/2/r_height:.10f} '
                      f'{(x2-x1)/r_width:.10f} {(y2-y1)/r_height:.10f}')
    (prepared / 'labels/val' / (stem + '.txt')).write_text('
'.join(labels))
# YOLO requires a train key in its schema; no training method is ever called.
yaml_path = prepared / 'data.yaml'
yaml_path.write_text(f'path: {prepared.as_posix()}
train: images/val
val: images/val
names:
  0: pothole
')
run_id = identity(dict(config_id=config_id, sample_id=manifest['sample_id'], model_sha256=model_digest))
write_json(OUT / 'run_identity.json', dict(run_id=run_id))
backup_tree(OUT, BACKUP)
print('Selected:', len(selected), '| pothole images:', POSITIVE_IMAGES,
      '| backgrounds:', BACKGROUND_IMAGES,
      '| total pothole boxes:', sum(len(r['boxes']) for r in selected))


## 5. Standard evaluation: mAP
mAP checks detection quality across confidence thresholds. It uses a low prediction
score floor (0.001), not the operating confidence of 0.25. Section 6 reports the
actual false alarms/misses at that operating confidence. These are distinct measures.

This pass is cached when complete. If interrupted, rerun it. Start with batch 4;
if GPU memory runs out, change only `batch=4` to `batch=1` here and rerun.


In [ ]:
map_file = OUT / 'map_metrics.json'
if not map_file.exists() and (BACKUP / map_file.name).exists():
    atomic_copy(BACKUP / map_file.name, map_file)
if map_file.exists():
    map_report = read_json(map_file)
    assert map_report['run_id'] == run_id, 'Cached evaluation belongs to another run.'
else:
    metrics = model.val(data=str(yaml_path), split='val', imgsz=IMAGE_SIZE,
                        batch=4, device=DEVICE, workers=0, conf=MAP_SCORE_FLOOR,
                        iou=0.7, max_det=300, half=False, plots=True,
                        project=str(ROOT / 'map_runs'), name='evaluation', exist_ok=True,
                        save_json=True, verbose=False)
    map_report = dict(run_id=run_id, map50=float(metrics.box.map50),
                      map50_95=float(metrics.box.map), speed_ms_per_image=metrics.speed,
                      device=str(DEVICE), note='Speed excludes download; not app latency.')
    # Preserve standard charts/JSON, without copying raw dataset or weights.
    for p in Path(metrics.save_dir).rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.json', '.csv', '.yaml'}:
            atomic_copy(p, OUT / 'standard_evaluation' / p.relative_to(metrics.save_dir))
    write_json(map_file, map_report)
backup_tree(OUT, BACKUP)
print(json.dumps(map_report, indent=2))


## 6. Fixed-confidence audit and example mistakes
A correct detection needs one-to-one overlap of at least 0.50 IoU with a labeled
pothole. Predictions are matched in descending confidence order. Extra boxes are
false positives; unmatched potholes are misses. Save each completed image to Drive
immediately. On reconnection rerun sections 1–4, then 5–6; saved images are skipped.

Green boxes = dataset labels. Red boxes = model predictions at confidence 0.25.
Examples include correct detections, false alarms and misses where available.


In [ ]:
def box_iou(a, b):
    intersection = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - intersection
    return intersection / union if union > 0 else 0.0

def match_boxes(truth, predictions, threshold):
    unmatched = set(range(len(truth)))
    tp = 0
    for p in sorted(predictions, key=lambda p: p['confidence'], reverse=True):
        candidates = [(box_iou(p['box'], truth[i]), i) for i in unmatched]
        overlap, best = max(candidates, default=(0.0, -1))
        if best >= 0 and overlap >= threshold:
            unmatched.remove(best)
            tp += 1
    return dict(tp=tp, fp=len(predictions)-tp, fn=len(truth)-tp)

prediction_dir = OUT / 'predictions'
prediction_dir.mkdir(exist_ok=True)
rows = []
for index, record in enumerate(selected):
    path = prediction_dir / f'{index:04d}.json'
    saved = BACKUP / 'predictions' / path.name
    if not path.exists() and saved.exists():
        atomic_copy(saved, path)
    if path.exists():
        row = read_json(path)
        assert row['run_id'] == run_id and row['id'] == record['id'], 'Prediction cache mismatch.'
    else:
        prediction = model.predict(str(ROOT / record['image']), imgsz=IMAGE_SIZE,
                                   conf=CONFIDENCE, iou=0.7, max_det=300,
                                   device=DEVICE, half=False, verbose=False)[0]
        boxes = prediction.boxes.xyxy.cpu().tolist()
        scores = prediction.boxes.conf.cpu().tolist()
        predictions = [dict(box=b, confidence=s) for b, s in zip(boxes, scores)]
        row = dict(run_id=run_id, id=record['id'], predictions=predictions,
                   truth=record['boxes'], **match_boxes(record['boxes'], predictions, MATCH_IOU))
        write_json(path, row)
        atomic_copy(path, saved)
    rows.append(row)
    if (index+1) % 25 == 0:
        print(index+1, '/', len(selected), 'saved')

tp, fp, fn = (sum(r[k] for r in rows) for k in ('tp', 'fp', 'fn'))
precision = tp/(tp+fp) if tp+fp else 0.0
recall = tp/(tp+fn) if tp+fn else 0.0
negative_rows = [r for r in rows if not r['truth']]
false_alarm_images = sum(r['fp'] > 0 for r in negative_rows)
report = dict(run_id=run_id, model=model_info, config=config, images=len(rows),
              potholes=sum(len(r['truth']) for r in rows), tp=tp, fp=fp, fn=fn,
              precision=precision, recall=recall,
              f1=2*precision*recall/(precision+recall) if precision+recall else 0.0,
              background_images=len(negative_rows), background_false_alarm_images=false_alarm_images,
              background_false_alarm_rate=false_alarm_images/len(negative_rows) if negative_rows else None,
              standard_evaluation=map_report,
              limitations=['Diagnostic enriched sample, not natural prevalence.',
                           'Training overlap and route independence not established.',
                           'RDD labels can miss defects; local camera validation still needed.'])
write_json(OUT / 'summary.json', report)
examples = OUT / 'examples'
examples.mkdir(exist_ok=True)
example_index = []
for kind, predicate in [('miss', lambda r: r['fn'] > 0),
                        ('false_alarm', lambda r: r['fp'] > 0),
                        ('correct', lambda r: r['tp'] > 0 and r['fp'] == 0 and r['fn'] == 0)]:
    indices = [i for i, row in enumerate(rows) if predicate(row)][:4]
    for i in indices:
        row, record = rows[i], selected[i]
        with Image.open(ROOT / record['image']) as source_image:
            image = source_image.convert('RGB')
        draw = ImageDraw.Draw(image)
        for box in row['truth']:
            draw.rectangle(box, outline='lime', width=3)
        for p in row['predictions']:
            draw.rectangle(p['box'], outline='red', width=2)
            draw.text((p['box'][0], p['box'][1]), f"{p['confidence']:.2f}", fill='red')
        name = f'{kind}_{i:04d}.jpg'
        ImageOps.contain(image, (1280, 960)).save(examples / name)
        example_index.append(dict(file=name, id=record['id'], tp=row['tp'], fp=row['fp'], fn=row['fn']))
write_json(OUT / 'example_index.json', example_index)
backup_tree(OUT, BACKUP)
print(f'Precision: {precision:.1%} | Recall: {recall:.1%} | Correct: {tp} | False boxes: {fp} | Missed: {fn}')
print(f'Background images with false alarms: {false_alarm_images}/{len(negative_rows)}')
from IPython.display import display
for item in example_index:
    print(item)
    with Image.open(examples / item['file']) as preview:
        display(ImageOps.contain(preview, (800, 500)))


## 7. Export and send the results
Download the ZIP and attach it in our chat. It includes summary scores, per-image
predictions, image manifest, settings, package versions, charts and example images.
It does not include the full dataset or model weights.

If the browser download fails, find the ZIP in your Drive run folder. Share the ZIP
and, if any step failed, its section number and error text. We will fix that existing
section instead of adding workaround cells. Scores alone do not establish deployment
readiness; we will review misses, false alarms and local camera examples together.


In [ ]:
assert (OUT / 'summary.json').exists(), 'Complete section 6 before exporting.'
backup_tree(OUT, BACKUP)
archive = Path(shutil.make_archive(str(ROOT / (RUN_NAME + '_results')), 'zip', OUT))
atomic_copy(archive, BACKUP / archive.name)
print('Saved ZIP:', BACKUP / archive.name)
from google.colab import files
files.download(str(archive))
